# 🔤 Devnagri — Evaluate & Export Trained Model

This notebook picks up after training is complete. It:
1. Loads the trained checkpoint from Google Drive
2. Evaluates on the test set (Word Accuracy + CER per language)
3. Converts to CTranslate2 (int8 quantized)
4. Downloads the optimized model

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install OpenNMT-py>=3.0 editdistance ctranslate2
!pip install "numpy<2"

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create local directories
!mkdir -p models configs results data/processed
print('✅ Drive mounted')

In [ ]:
# Copy data from Google Drive
!cp /content/drive/MyDrive/devnagri/data/processed/*.txt data/processed/

# Verify
!echo '--- Data Files ---'
!wc -l data/processed/*.txt

In [ ]:
# Find trained checkpoints on Google Drive
import glob

checkpoints = sorted(glob.glob('/content/drive/MyDrive/devnagri/models/transliteration_model_step_*.pt'))
print(f'Found {len(checkpoints)} checkpoints:')
for cp in checkpoints:
    print(f'  {cp}')

latest = checkpoints[-1] if checkpoints else None
print(f'\n✅ Using: {latest}')

## 2. Evaluate on Test Set

In [ ]:
# Set latest checkpoint as shell variable
import os
os.environ['LATEST_MODEL'] = latest
print(f'Model: {latest}')

In [ ]:
# Run translation on test set
!python -m onmt.bin.translate \
    -model $LATEST_MODEL \
    -src data/processed/src-test.txt \
    -output results/predictions.txt \
    -beam_size 5 \
    -replace_unk \
    -gpu 0

In [ ]:
# Compute evaluation metrics (Word Accuracy + CER per language)
import editdistance
from collections import defaultdict
import json

def chars_to_word(s):
    return s.replace(' ', '').strip()

def get_lang(s):
    if s.startswith('<') and '>' in s:
        return s[1:s.index('>')]
    return 'unknown'

with open('data/processed/src-test.txt') as f:
    sources = [l.strip() for l in f]
with open('data/processed/tgt-test.txt') as f:
    references = [l.strip() for l in f]
with open('results/predictions.txt') as f:
    predictions = [l.strip() for l in f]

per_lang = defaultdict(lambda: {'total': 0, 'correct': 0, 'cer_dist': 0, 'cer_len': 0})

for src, ref, pred in zip(sources, references, predictions):
    lang = get_lang(src)
    p = chars_to_word(pred)
    r = chars_to_word(ref)
    s = per_lang[lang]
    s['total'] += 1
    if p == r: s['correct'] += 1
    s['cer_dist'] += editdistance.eval(list(p), list(r))
    s['cer_len'] += len(r)

print(f"{'Lang':<10} {'Pairs':>8} {'Acc%':>10} {'CER%':>10}")
print('=' * 40)
total_correct = 0
total_pairs = 0
total_cer_dist = 0
total_cer_len = 0

for lang, s in sorted(per_lang.items()):
    acc = s['correct']/s['total']*100
    cer = s['cer_dist']/s['cer_len']*100
    print(f"{lang:<10} {s['total']:>8} {acc:>9.2f} {cer:>9.2f}")
    total_correct += s['correct']
    total_pairs += s['total']
    total_cer_dist += s['cer_dist']
    total_cer_len += s['cer_len']

print('-' * 40)
overall_acc = total_correct/total_pairs*100
overall_cer = total_cer_dist/total_cer_len*100
print(f"{'OVERALL':<10} {total_pairs:>8} {overall_acc:>9.2f} {overall_cer:>9.2f}")

# Save results
results = {
    'per_language': {lang: {'accuracy': s['correct']/s['total']*100, 'cer': s['cer_dist']/s['cer_len']*100, 'total': s['total']} for lang, s in per_lang.items()},
    'overall': {'accuracy': overall_acc, 'cer': overall_cer, 'total': total_pairs}
}
with open('results/eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'\n✅ Results saved to results/eval_results.json')

In [ ]:
# Show sample predictions
print(f"{'Source':<30} {'Expected':<20} {'Predicted':<20} {'Match'}")
print('=' * 80)
for i in range(0, min(30, len(sources)), 1):
    src_word = chars_to_word(sources[i].split('> ')[-1]) if '> ' in sources[i] else sources[i]
    lang = get_lang(sources[i])
    ref_word = chars_to_word(references[i])
    pred_word = chars_to_word(predictions[i])
    match = '✅' if ref_word == pred_word else '❌'
    print(f"[{lang}] {src_word:<25} {ref_word:<20} {pred_word:<20} {match}")

## 3. Convert to CTranslate2

In [ ]:
# Convert to CTranslate2 with int8 quantization
!ct2-opennmt-py-converter \
    --model_path $LATEST_MODEL \
    --output_dir models/ct2_model \
    --quantization int8

print('\n✅ CTranslate2 model created at models/ct2_model/')
!ls -la models/ct2_model/

In [ ]:
# Quick CTranslate2 inference test
import ctranslate2

translator = ctranslate2.Translator('models/ct2_model', device='cpu')

test_words = [
    ('<hin>', 'namaste'),
    ('<hin>', 'bharat'),
    ('<hin>', 'delhi'),
    ('<ben>', 'kolkata'),
    ('<ben>', 'dhanyabad'),
    ('<tam>', 'chennai'),
    ('<tam>', 'vanakkam'),
]

print(f"{'Input':<20} {'Output'}")
print('=' * 40)
for prefix, word in test_words:
    tokens = [prefix] + list(word)
    result = translator.translate_batch([tokens], beam_size=5)
    output = ''.join(result[0].hypotheses[0])
    lang = prefix[1:-1]
    print(f"[{lang}] {word:<15} → {output}")

## 4. Save & Download

In [ ]:
# Save CT2 model and results to Google Drive
!cp -r models/ct2_model /content/drive/MyDrive/devnagri/models/
!cp -r results /content/drive/MyDrive/devnagri/
print('✅ CT2 model and results saved to Google Drive!')

In [ ]:
# Download zip files to local machine
!zip -r ct2_model_download.zip models/ct2_model/
!zip -r results_download.zip results/

from google.colab import files
files.download('ct2_model_download.zip')
files.download('results_download.zip')
print('\n✅ Downloads started!')

In [ ]:
# Also download the best .pt checkpoint
import shutil
import os

# Copy checkpoint locally for zipping
checkpoint_name = os.path.basename(latest)
shutil.copy(latest, checkpoint_name)
!zip checkpoint_download.zip $checkpoint_name

from google.colab import files
files.download('checkpoint_download.zip')
print(f'\n✅ Checkpoint {checkpoint_name} download started!')